# Demo C1 — Final Project: Document Research Agent

Capstone project building on `Demo_C1.ipynb` — instead of separate feature demos, this notebook composes several of them into one working agent, built with the current (v1) LangChain/LangGraph APIs. See `CLAUDE.md` at the repo root for the version-specific gotchas already worked out (`create_agent` shape, the `chromadb` pin, `langchain_classic`, etc.).

**Roadmap** — built one stage at a time, each verified before moving to the next:

1. **Knowledge base** — load a PDF, split it, embed it, build a retriever. Tested standalone.
2. **`get_context` tool** — wrap the retriever as an agent tool. Tested standalone.
3. **Compose tool** — a multi-step LCEL chain (the `RunnablePassthrough.assign` pattern from `Demo_C1`) exposed as a second tool.
4. **Agent, one tool** — wire `get_context` into `create_agent`.
5. **Agent, both tools** — add the compose tool.
6. **Memory** — add a `checkpointer` so the agent remembers earlier turns.
7. **Structured output** — add a `response_format` Pydantic model.

Only stage 1 is scaffolded below. Later stages get added once this one is working.

## Setup

In [1]:
import json
import os
import warnings
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore, LocalFileStore, create_kv_docstore
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field

warnings.filterwarnings("ignore")
load_dotenv()

/var/folders/pf/7lwsqjw92g96dl5sfdckf9_r0000gn/T/ipykernel_2354/1480796897.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [2]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

In [3]:
model = ChatOpenAI(model="gpt-4o-mini", api_key=os.environ.get("OPENAI_API_KEY"))
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small", api_key=os.environ.get("OPENAI_API_KEY")
)

PDF_PATH = "../../data/O-príncipe-Nicolau-Maquiavel.pdf"  # same source Demo_C1 already loads
CHROMA_PERSIST_DIR = (
    "../../data/chroma_demo_c1_project"  # persisted so we don't re-embed every restart
)

## 1. Knowledge base

Build the retrieval pipeline from `Demo_C1`'s DOCUMENT / TEXT SPLITTERS / EMBEDDINGS / VECTOR STORES / RETRIEVERS sections, combined into one `ParentDocumentRetriever` (imported from `langchain_classic.retrievers` — see `CLAUDE.md` for why).

You already built this exact pattern in `Demo_C1.ipynb`'s **PARENT DOCUMENT RETRIEVER** section — that's your reference, not this notebook.

Once it runs: test with `retriever.invoke("some query")` and actually look at what comes back — is it a small chunk or the full parent document? That answer matters once this becomes a tool the agent calls, so don't skip inspecting it.

In [4]:
#
# * HOW TO CONSULT FOR EXISTING COLLECTIONS

# // 1 - From an existing vector_store
# vector_store._client.list_collections()

# // 2 - Standalone, without instantiating a langchain Chroma object
# import chromadb

# client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
# client.list_collections()


# * HOW TO DELETE COLLECTIONS

# // 1 - From instantiated Chroma
# vector_store.delete_collection()

# // 2 - From a Standalone
# import chromadb
# client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
# client.delete_collection(name="C1_project")


Once this stage runs and `test_results` looks right to you, let's check in before scaffolding stage 2 (the `get_context` tool).

### What to measure here ?  
> Did the parent-lookup step fired ?  
>> compare the result against the child:
>>> Close to 400 → you're likely still getting child-level content back, not parents. 
>>>> Meaningfully bigger than 400 → the parent lookup did its job. 

In [5]:
# child_matches[i].page_content → the actual child chunk that matched in Chroma (small, ~400 chars)

# child_matches[i].metadata["doc_id"] → the in-memory id, i.e. the exact key test_results[i] was fetched by from docstore

Implementing the pdf concatenation.

In [6]:
# TODO: load the PDF at PDF_PATH
# HINT: PyPDFLoader(...).load() returns a list[Document]
pdf_docs = PyPDFLoader(PDF_PATH)
pdf_loader = pdf_docs.load()
len(pdf_loader)
pdf_loader[0]

154

Document(metadata={'producer': 'calibre (5.39.1) [https://calibre-ebook.com]', 'creator': 'calibre (5.39.1) [https://calibre-ebook.com]', 'creationdate': '2022-03-13T01:35:03+00:00', 'author': 'Nicolau Maquiavel', 'keywords': 'Political Science, History & Theory', 'moddate': '2022-03-29T14:54:02-03:00', 'title': 'O príncipe', 'source': '../../data/O-príncipe-Nicolau-Maquiavel.pdf', 'total_pages': 154, 'page': 0, 'page_label': '1'}, page_content='')

In [7]:
def concat(docs: list[Document]) -> str:
    return "\n".join(i.page_content for i in docs)


pdf_concat = concat(pdf_loader)
len(pdf_concat)
type(pdf_concat)
pdf_concat[:50]

282670

str

'\nO PRÍNCIPE\nNICOLAU MAQUIAVEL nasceu em Florença e'

In [8]:
new_metadata = {k: v for k, v in pdf_loader[0].metadata.items()}
del new_metadata["page"]
del new_metadata["page_label"]
new_metadata

{'producer': 'calibre (5.39.1) [https://calibre-ebook.com]',
 'creator': 'calibre (5.39.1) [https://calibre-ebook.com]',
 'creationdate': '2022-03-13T01:35:03+00:00',
 'author': 'Nicolau Maquiavel',
 'keywords': 'Political Science, History & Theory',
 'moddate': '2022-03-29T14:54:02-03:00',
 'title': 'O príncipe',
 'source': '../../data/O-príncipe-Nicolau-Maquiavel.pdf',
 'total_pages': 154}

In [9]:
pdf_concat_document = Document(page_content=pdf_concat, metadata=new_metadata)
type(pdf_concat_document)

langchain_core.documents.base.Document

In [10]:
# TODO: define a child_splitter (small chunks, for the vector index) and a
# parent_splitter (larger chunks, for full context) — ParentDocumentRetriever needs both
child_splitter = CharacterTextSplitter(separator="\n", chunk_size=400, chunk_overlap=50)
parent_splitter = CharacterTextSplitter(separator="\n", chunk_size=4000, chunk_overlap=100)

# TODO: create a persisted Chroma vector store (for child chunks) and an
# InMemoryStore (for parent docs), then assemble the ParentDocumentRetriever
# HINT: from langchain_classic.storage import InMemoryStore
# HINT: from langchain_classic.retrievers import ParentDocumentRetriever
# HINT: Chroma(..., persist_directory=CHROMA_PERSIST_DIR, embedding_function=embedding_model)
vector_store = Chroma(
    collection_name="C1_project",
    persist_directory=CHROMA_PERSIST_DIR,
    embedding_function=embedding_model,
)
docstore = create_kv_docstore(LocalFileStore("../../data/docstore_c1_project"))

retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# TODO: index pdf_docs into the retriever
if vector_store._collection.count() == 0:
    retriever.add_documents([pdf_concat_document])
else:
    print(f"Skipping — collection already has {vector_store._collection.count()} entries.")

# TODO: test it — inspect what actually comes back, not just that it ran
test_results = retriever.invoke("qual o nome do livro?")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Skipping — collection already has 807 entries.


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [11]:
# vector_store.delete_collection()
vector_store._client.list_collections()


# * why is that under vector_store it finds a collection that was instantiated by vector_store_v2 ?
# _client is the underlying chromadb client, and that client is scoped to the persist directory (CHROMA_PERSIST_DIR), not to any single Chroma instance or collection. So, both instances open a client pointed at the same on-disk SQLite/store at CHROMA_PERSIST_DIR.

[Collection(name=C1_project)]

In [12]:
child_matches = vector_store.similarity_search("qual o nome do livro?")

In [13]:
type(child_matches)
len(child_matches)
type(child_matches[0])
len(child_matches[0].page_content)
child_matches[0].page_content

list

4

langchain_core.documents.base.Document

390

'trademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W arrak\nTRADUÇÃO DOS APÊNDICES\nLuiz A. de Araújo\nPREPARAÇÃO\nSilvia Maximini Félix\nREVISÃO\nAna Maria Barbosa\nHuendel V iana'

In [14]:
results_metadata = pd.DataFrame(
    [{"doc_id": i.metadata["doc_id"], "child_page_content": i.page_content} for i in child_matches]
)
results_metadata

,doc_id,child_page_content
0,8bf0bf7d-177e-4bcf-af58-fb7d059b6565,trademarks of Penguin Books Limited and/or Pen...
1,8bf0bf7d-177e-4bcf-af58-fb7d059b6565,REVISÃO\nAna Maria Barbosa\nHuendel V iana\nIS...
2,aa267e52-7eb9-4864-9200-240ba39bd645,XXV. Em que medida a fortuna controla as coisa...
3,aa267e52-7eb9-4864-9200-240ba39bd645,Classics. Faleceu no dia 6 de abril de 2001.\n...


In [15]:
results_metadata["child_page_content"][0]
results_metadata["doc_id"][0]

'trademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W arrak\nTRADUÇÃO DOS APÊNDICES\nLuiz A. de Araújo\nPREPARAÇÃO\nSilvia Maximini Félix\nREVISÃO\nAna Maria Barbosa\nHuendel V iana'

'8bf0bf7d-177e-4bcf-af58-fb7d059b6565'

In [16]:
len(test_results[0].page_content)
test_results[0].page_content

len(child_matches[0].page_content)
child_matches[0].page_content

len(docstore.mget([results_metadata["doc_id"][0]])[0].page_content)
docstore.mget([results_metadata["doc_id"][0]])[0].page_content

1334

'JOLY, M. Diálogo no inferno entre Maquiavel e Montesquieu , Unesp, 2009.\nSKINNER, Q. As fundações do pensamento político moderno , São Paulo, Companhia das\nLetras, 1989.\nVIROLI, M. O sorriso de Nicolau: história de Maquiavel , São Paulo, Estação Liberdade, 2002.\nWHITE, M. Maquiavel, um homem incompreendido , Rio de Janeiro, Record, 2007.\nCopyright das notas © 1961, 1975, 1981, 1995, 1999 by George Bull\nCopyright da introdução © 1999 by Anthony Grafton\nCopyright do prefácio © 2010 by Fernando Henrique Cardoso\nGrafia atualizada segundo o Acordo Ortográfico da Língua Portuguesa de 1990, que entrou\nem vigor no Brasil em 2009.\nPenguin and the associated logo and trade dress are registered and/or unregistered\ntrademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W

390

'trademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W arrak\nTRADUÇÃO DOS APÊNDICES\nLuiz A. de Araújo\nPREPARAÇÃO\nSilvia Maximini Félix\nREVISÃO\nAna Maria Barbosa\nHuendel V iana'

1334

'JOLY, M. Diálogo no inferno entre Maquiavel e Montesquieu , Unesp, 2009.\nSKINNER, Q. As fundações do pensamento político moderno , São Paulo, Companhia das\nLetras, 1989.\nVIROLI, M. O sorriso de Nicolau: história de Maquiavel , São Paulo, Estação Liberdade, 2002.\nWHITE, M. Maquiavel, um homem incompreendido , Rio de Janeiro, Record, 2007.\nCopyright das notas © 1961, 1975, 1981, 1995, 1999 by George Bull\nCopyright da introdução © 1999 by Anthony Grafton\nCopyright do prefácio © 2010 by Fernando Henrique Cardoso\nGrafia atualizada segundo o Acordo Ortográfico da Língua Portuguesa de 1990, que entrou\nem vigor no Brasil em 2009.\nPenguin and the associated logo and trade dress are registered and/or unregistered\ntrademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W

## 2. `get_context` tool

Wrap `retriever` (the V2 one, built on `pdf_concat_document`) as an agent tool, using the `@tool` decorator pattern from `Demo_C1`'s **Custom tool via the `@tool` decorator** section — that's your reference, not this notebook.

A few things to think about before you write it:

- The function needs a type-annotated argument (the search query) and a docstring — `@tool` builds the tool's schema and description from those, so the docstring is what the agent reads to decide when to call this.
- `retriever.invoke(query)` returns `list[Document]`. Tools generally return strings back to the agent — so this function has to reduce that list down to something text-based. What's lost if you just join `page_content` with no separator? What do you gain by including each source's metadata alongside its content?
- Per `CLAUDE.md`: tool `name` must match `^[a-zA-Z0-9_-]+$` — no spaces. `@tool` uses the function name by default, so name the function accordingly.

Test it standalone by calling the function directly (not through an agent) before moving on — same rule as stage 1.

In [17]:
# TODO: define get_context(query: str) -> str
# - call retriever.invoke(query)
# - reduce the resulting list[Document] into a single string to return
# - write a docstring the agent will use to decide when to call this tool


@tool
def get_context(query: str) -> str:
    """
    Use this to retrieve documents when a question is related to a book called "O príncipe" by Nicolau Maquiavel.
    """
    doc = retriever.invoke(query)
    return "\n".join(i.page_content for i in doc)  # reduce Document into a single str


In [18]:
# TODO: test standalone — call the underlying function directly (e.g. get_context.invoke("...")
# or get_context.func("...") depending on how you call a @tool-wrapped function) and read the output
get_context.func("o que Maquiavel diz sobre a fortuna e a virtù")[:300]

'sentia mais seguro sobre o que mais convinha à França para conter\nos espanhóis ao sul. Mais seguro, mostrou-se mais cordial, menos\narrogante, soube dissimular sua força e seus instintos. Sabia que\ndentre chefes de outras cidades, que também se apressaram a\nvisitar Luís XII buscando alianças, e até m'

## 3. get_answer tool

Build a second tool the agent can call — one that doesn't just return raw excerpts like `get_context`, but composes retrieval *and* generation into a single synthesized answer. Use the multi-step `RunnablePassthrough.assign` pattern from `Demo_C1`'s **LCEL** section (the `location_chain_lcel` → `dish_chain_lcel` → `time_chain_lcel` composition, combined via `overall_chain_lcel`) as your reference — that's your template for "thread new keys through a pipeline," applied here to retrieval + answering instead of location/dish/time.

The shape you're building toward:

1. Start from `{"question": "..."}`.
2. `RunnablePassthrough.assign(context=...)` — call the retriever (or reuse `get_context.func`) to fetch context for `x["question"]`, adding it under a `"context"` key while keeping `"question"` intact.
3. `RunnablePassthrough.assign(answer=...)` — feed `{"context": ..., "question": ...}` into a `prompt | model | StrOutputParser()` chain to generate the final answer, added under `"answer"`.

Questions worth thinking through before you start:
- What's actually different about what this tool gives the agent, compared to `get_context`? When might the agent reach for one over the other?
- Your prompt needs `input_variables` matching the keys you assign (`context`, `question`, or whatever names you pick) — a mismatch here fails silently or errors deep in the chain, so keep the naming consistent across cells.

Test each piece standalone — the answer chain alone, then the full composed chain — before wrapping any of it as a `@tool`. Same discipline as stages 1 and 2: isolate failures before they get buried inside an agent's reasoning loop.

In [19]:
# TODO: define a PromptTemplate with input_variables=["context", "question"]
# (or whatever keys you settle on — just stay consistent through the rest of this stage)

template = """You are a political theory analyst specializing in Nicolau Maquiavel's "O Príncipe". Given the context excerpts below, provide an interpretive analysis — explain the reasoning, implications, or significance behind what the text says. Don't just restate the excerpts; ground your interpretation in them.

Context:
{context}

Question: {question}

Your analysis:"""

prompt = PromptTemplate(input_variables=["question", "context"], template=template)

# TODO: compose the answer chain: prompt | model | StrOutputParser()
answer_chain = prompt | model | StrOutputParser()

In [20]:
# TODO: test answer_chain standalone with a hand-written context — before wiring in retrieval.
# This isolates prompt/model/parser bugs from retriever bugs, testing whether the prompt successfully tells it "answer from this text, not from your own memory." The expected result is a context grounded llm response.

answer_chain.invoke({"question": "como esta o rei?", "context": "o rei esta nu"})[:300]


'The phrase "o rei está nu," which translates to "the king is naked," serves as a powerful metaphor within the framework of political theory, particularly in the context of Machiavelli\'s "O Príncipe" (The Prince). This saying encapsulates a profound truth about the nature of power, leadership, and th'

In [21]:
# TODO: build compose_chain with RunnablePassthrough.assign, two steps:
# 1. assign "context" — call the retriever (or get_context.func) for x["question"]
# 2. assign "answer"  — call answer_chain.invoke() using the context + question from step 1
# HINT: see Demo_C1's overall_chain_lcel (location -> meal -> recipe -> time) for the exact shape


compose_chain = RunnablePassthrough.assign(
    context=lambda x: get_context.func(x["question"])
) | RunnablePassthrough.assign(
    answer=lambda x: answer_chain.invoke({"question": x["question"], "context": x["context"]})
)

In [22]:
# TODO: test the full chain: compose_chain.invoke({"question": "..."})
# Check: does result["answer"] actually look grounded in result["context"],
# or does it read like the model ignoring context and answering from general knowledge?
question_1 = "Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, incluindo a morte de Ramiro d'Orco?"

question_2 = "O que Maquiavel diz sobre o papel da fortuna na vida do príncipe, e ele usa alguma metáfora para isso?"


result = compose_chain.invoke({"question": question_1})
context_1 = result["context"]

In [23]:
type(result)
result.keys()

dict

dict_keys(['question', 'context', 'answer'])

In [24]:
result["question"]
result["context"][:300]
result["answer"][:300]

"Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, incluindo a morte de Ramiro d'Orco?"

'sentia mais seguro sobre o que mais convinha à França para conter\nos espanhóis ao sul. Mais seguro, mostrou-se mais cordial, menos\narrogante, soube dissimular sua força e seus instintos. Sabia que\ndentre chefes de outras cidades, que também se apressaram a\nvisitar Luís XII buscando alianças, e até m'

'A análise das ações de César Bórgia, conforme relatadas por Maquiavel em "O Príncipe", revela uma complexa interdependência entre a crueldade e a eficácia política, uma das temáticas centrais na obra do pensador florentino. Não se trata apenas de um elogio à brutalidade em si, mas da maneira como es'

In [25]:
# TODO: wrap compose_chain as a second tool
# - needs a type-annotated query argument and a docstring the agent will read to decide when to call it
# - decide what to return to the agent: just the answer string, or something richer?
# - name it something distinct from get_context — per CLAUDE.md, name must match ^[a-zA-Z0-9_-]+$


@tool
def get_answer(query: str) -> dict:
    """Use this to get an answer when a question is related to a book called "O príncipe" by Nicolau Maquiavel."""

    response = compose_chain.invoke({"question": query})
    return response

In [26]:
# TODO: test standalone (.func(...) or .invoke(...)) before this gets wired into the agent in stage 4
result = get_answer.func(question_1)


In [27]:
type(result)
result.keys()
result["question"]
result["context"][:300]
result["answer"][:300]

dict

dict_keys(['question', 'context', 'answer'])

"Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, incluindo a morte de Ramiro d'Orco?"

'sentia mais seguro sobre o que mais convinha à França para conter\nos espanhóis ao sul. Mais seguro, mostrou-se mais cordial, menos\narrogante, soube dissimular sua força e seus instintos. Sabia que\ndentre chefes de outras cidades, que também se apressaram a\nvisitar Luís XII buscando alianças, e até m'

"Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, incluindo a morte de Ramiro d'Orco, por várias razões que refletem a complexidade de sua teoria política, bem como a sua ênfase na eficácia e na necessidade pragmática de um líder para manter o poder.\n\nPrimeirame"

## 4. Agent, one tool

Wire `get_context` into an agent using `create_agent` — see `Demo_C1`'s **AGENT** section for the reference pattern, and `CLAUDE.md`'s v1 API notes at the repo root for the specifics already worked out for this stack:

- Import from `langchain.agents`, not `langgraph.prebuilt.create_react_agent` (deprecated in `langgraph==1.0`).
- `system_prompt` is a plain `str` (or `SystemMessage`) — no callable/dynamic prompt hook like the old `create_react_agent`'s `prompt` param.
- No memory yet (`checkpointer` is stage 6) and no structured output yet (`response_format` is stage 7) — keep this stage to just: one model, one tool, one system prompt.

Two things worth thinking about before you write the `system_prompt`:
- What should the agent do when a question has nothing to do with the book? (`get_context`'s docstring already tells the agent *when* to call it — but the system prompt shapes the agent's overall behavior, including when *not* to call any tool.)
- `create_agent` returns a compiled graph, not a plain chain — check `Demo_C1`'s AGENT section for the actual shape of `.invoke(...)`'s input/output (hint: it's message-based, not a plain string in/out like `answer_chain`).

Test by invoking the agent directly with a question you already know `get_context` handles well (reuse one of your earlier validated questions) — and don't just check the final answer text. Inspect the full message list in the response: did the agent actually *call* the tool, or did it answer from its own knowledge without touching your retriever at all? That distinction matters — an agent that never calls the tool would still produce a plausible-sounding answer, exactly the kind of "looks grounded but isn't" trap you already ran into with `compose_chain`.

In [28]:
# TODO: build the agent
# - model=model
# - tools=[get_context]
# - system_prompt=... (plain str — decide what it should say about scope/behavior)
agent = create_agent(
    model=model,
    tools=[get_context],
    system_prompt="""
    You are a research agent for the book called "O Príncipe" by Nicolau Maquiavel. When a question is related to this book, use get_context to retrieve documents as a context. If the question is not related with this book, answer it by your own knowledge without using the tools.
""",
)

In [29]:
# TODO: invoke the agent with a question you already know get_context handles well
# HINT: check Demo_C1's AGENT section for the exact input/output shape (message-based,
# not a plain string like answer_chain)
# agent_result = agent.invoke({"messages": [{"role": "user", "content": question_1}]})
agent_result = agent.invoke({"messages": [HumanMessage(content=question_1)]})

In [30]:
type(agent_result)
agent_result.keys()
type(agent_result["messages"])
len(agent_result["messages"])
[i.type for i in agent_result["messages"]]

dict

dict_keys(['messages'])

list

4

['human', 'ai', 'tool', 'ai']

In [31]:
# TODO: inspect the full message list in agent_result — don't just read the final answer.
# Look for a ToolMessage (or an AIMessage with tool_calls) confirming get_context
# actually got invoked, not just a plausible-sounding answer from the model's own knowledge.
# agent_result

agent_result["messages"][0].content
agent_result["messages"][1].tool_calls
agent_result["messages"][2].content[:100]
agent_result["messages"][3].content[:100]

"Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, incluindo a morte de Ramiro d'Orco?"

[{'name': 'get_context',
  'args': {'query': "César Bórgia e a morte de Ramiro d'Orco"},
  'id': 'call_sEqBYOh2RgLX6voj80vusWpx',
  'type': 'tool_call'}]

'O leitor interessado num relato sumário das circunstâncias e dos antecedentes históricos\ngerais do p'

"Nicolau Maquiavel elogia as ações de César Bórgia, em particular a morte de Ramiro d'Orco, na conqui"

## 5. Agent, both tools

Add `get_answer` alongside `get_context`, so the agent has two tools that can both answer questions about the book — but do different things:

- `get_context` returns raw retrieved excerpts — the agent still has to read and synthesize them itself.
- `get_answer` already runs `compose_chain` — the agent gets back a finished, synthesized answer, not raw material.

Questions worth thinking through before you touch the system prompt:

- Given both tools can answer book questions, how should the agent choose between them? Is each tool's docstring enough signal on its own, or does the system prompt need to say something explicit about when to prefer one over the other?
- Both tools now return a plain `str` to the agent — `get_answer` intentionally drops the `context`/`question` fields `compose_chain` computes internally, so the agent only ever sees the synthesized answer, not the excerpts it was grounded in.

Test with a couple of questions and inspect the message list — same discipline as stage 4. Check *which* tool actually got called (and whether that matches what you'd expect for that question), not just whether you got a plausible-looking answer.

In [32]:
# TODO: build agent_v2 with both tools
# - tools=[get_context, get_answer]
# - system_prompt=... — decide whether it needs to say anything explicit about
#   choosing between the two tools, or whether the two docstrings are enough on their own

agent_v2 = create_agent(
    model=model,
    tools=[get_context, get_answer],
    system_prompt="""You are a research agent for the book "O Príncipe" by Nicolau Maquiavel.

- If the question asks what the book says about something (a fact, a definition, a passage) — use get_context to return excerpts directly from the text.
- If the question asks for analysis, interpretation, or opinion about something in the book (why, what does it mean, what's the significance) — use get_answer, which synthesizes an interpretive answer grounded in the book's content.
- get_context only retrieves, so it's cheaper; get_answer also generates an analysis, so it costs more — prefer get_context when a plain excerpt would fully answer the question.
- If the question is unrelated to this book, answer from your own knowledge without using either tool.
""",
)

In [33]:
# TODO: invoke agent_v2 with a question you already know a book-related tool handles well
# (reuse question_1 or question_2, or write a new one) and inspect the message list
# — same shape as stage 4's agent_result
result = agent_v2.invoke({"messages": [HumanMessage(content=question_1)]})


In [34]:
# TODO: same inspection discipline as stage 4 — find the AIMessage(s) with tool_calls,
# check tool_calls[0]["name"] to see which tool actually got called, and whether that
# matches what you'd expect given the question you asked
type(result)
result.keys()
type(result["messages"])
len(result["messages"])
[i.type for i in result["messages"]]

dict

dict_keys(['messages'])

list

4

['human', 'ai', 'tool', 'ai']

In [56]:
json.loads(result["messages"][2].content)["context"][:100]

'sentia mais seguro sobre o que mais convinha à França para conter\nos espanhóis ao sul. Mais seguro, '

In [36]:
result["messages"][0].content
result["messages"][1].tool_calls
print(result["messages"][2].content[:500])
result["messages"][3].content[:100]

"Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, incluindo a morte de Ramiro d'Orco?"

[{'name': 'get_answer',
  'args': {'query': "Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, incluindo a morte de Ramiro d'Orco?"},
  'id': 'call_qborXbbLsmGnbLYihE1525JV',
  'type': 'tool_call'}]

{"question": "Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, incluindo a morte de Ramiro d'Orco?", "context": "sentia mais seguro sobre o que mais convinha à França para conter\nos espanhóis ao sul. Mais seguro, mostrou-se mais cordial, menos\narrogante, soube dissimular sua força e seus instintos. Sabia que\ndentre chefes de outras cidades, que também se apressaram a\nvisitar Luís XII buscando alianças, e até mesmo entre seus capitães,\ncorria so


'Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, assim como a'

## 6. Memory

Add a `checkpointer` so `agent_v2` remembers earlier turns in a conversation, instead of treating every `.invoke()` as a fresh start.

**Heads up: `Demo_C1`'s `# MEMORY` section is not your reference here.** It builds `InMemoryChatMessageHistory` + `RunnableWithMessageHistory` — the pre-v1 pattern. Per `CLAUDE.md`, `RunnableWithMessageHistory` only wraps `Runnable` chains, and `create_agent` returns a compiled *graph*, not a chain — so that pattern doesn't apply. The v1-native replacement, already worked out in `CLAUDE.md`:

- Pass `checkpointer=InMemorySaver()` (from `langgraph.checkpoint.memory`) to `create_agent`.
- Address a specific conversation via `config={"configurable": {"thread_id": ...}}` on each `.invoke()` call.

The shape you're building toward: `agent_v3` — same `model`/`tools`/`system_prompt` as `agent_v2`, just with a `checkpointer` added.

Questions worth thinking through before you test it:
- When you `.invoke()` twice with the *same* `thread_id`, does the checkpointer automatically feed the earlier turn's messages back into the model, or do you still have to pass the full message history yourself each call?
- What should happen if you `.invoke()` with a *different* `thread_id`? Should that conversation know anything about the first one?
- In a notebook, `thread_id` is just a string you make up. In a real app, where would it realistically come from?

Test plan: run two turns on the *same* `thread_id`, where the second turn only makes sense if the agent remembers the first (e.g. refer back to "isso"/"essa ação" instead of restating the subject). Then run that same follow-up question on a *different* `thread_id` and see what happens without the missing context — that's your proof the isolation is real, not just that memory exists.

In [37]:
# TODO: build agent_v3 — same model/tools/system_prompt as agent_v2, plus checkpointer=InMemorySaver()
# HINT: reuse agent_v2's system_prompt string rather than retyping it

agent_v3 = create_agent(
    model=model,
    tools=[get_context, get_answer],
    checkpointer=InMemorySaver(),
    system_prompt="""You are a research agent for the book "O Príncipe" by Nicolau Maquiavel.

- If the question asks what the book says about something (a fact, a definition, a passage) — use get_context to return excerpts directly from the text.
- If the question asks for analysis, interpretation, or opinion about something in the book (why, what does it mean, what's the significance) — use get_answer, which synthesizes an interpretive answer grounded in the book's content.
- get_context only retrieves, so it's cheaper; get_answer also generates an analysis, so it costs more — prefer get_context when a plain excerpt would fully answer the question.
- If the question is unrelated to this book, answer from your own knowledge without using either tool.
""",
)

In [38]:
# TODO: turn 1 — invoke agent_v3 with a book question, passing config={"configurable": {"thread_id": thread_1}}
# thread_1 = "..."
# turn_1 = agent_v3.invoke({"messages": [HumanMessage(content=...)]}, config={"configurable": {"thread_id": thread_1}})
# inspect turn_1["messages"] — same discipline as stages 4/5

thread_1 = "maquiavel-thread-1"
turn_1 = agent_v3.invoke(
    {"messages": [HumanMessage(content=question_1)]},
    config={"configurable": {"thread_id": thread_1}},
)

In [39]:
type(turn_1)
turn_1.keys()
[i.type for i in turn_1["messages"]]

dict

dict_keys(['messages'])

['human', 'ai', 'tool', 'ai']

In [40]:
turn_1["messages"][0].content
turn_1["messages"][1].tool_calls
turn_1["messages"][2].content[:100]
turn_1["messages"][3].content[:100]

"Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, incluindo a morte de Ramiro d'Orco?"

[{'name': 'get_answer',
  'args': {'query': "Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, incluindo a morte de Ramiro d'Orco?"},
  'id': 'call_FC8QTOXHEApYl6a8j2OmLO7d',
  'type': 'tool_call'}]

'{"question": "Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da'

'Maquiavel elogia as ações de César Bórgia na conquista e pacificação da Romanha, incluindo a execuçã'

In [41]:
# TODO: turn 2 — same thread_id, a follow-up question that ONLY makes sense if the agent
# remembers turn_1 (don't restate the subject — refer back to "isso"/"essa ação"/etc.)
# turn_2 = agent_v3.invoke({"messages": [HumanMessage(content=...)]}, config={"configurable": {"thread_id": thread_1}})
# Check turn_2["messages"] — how many messages are in the list now? Does it include turn_1's
# messages too, or just this turn's? That answers "does the checkpointer feed history back in."

question_1_turn_2 = "E isso não contradiz o que ele diz em outros trechos ?"


turn_2 = agent_v3.invoke(
    {"messages": [HumanMessage(content=question_1_turn_2)]},
    config={"configurable": {"thread_id": thread_1}},
)

In [42]:
type(turn_2["messages"])
len(turn_2["messages"])  # double the items, from 4 to 8

list

8

In [43]:
[i.type for i in turn_2["messages"]]

['human', 'ai', 'tool', 'ai', 'human', 'ai', 'tool', 'ai']

In [44]:
# TODO: isolation check — ask the SAME follow-up from turn_2, but with a DIFFERENT thread_id
# thread_2 = "..."
# turn_2_isolated = agent_v3.invoke({"messages": [HumanMessage(content=...)]}, config={"configurable": {"thread_id": thread_2}})
# Compare this answer to turn_2's — does the agent now ask for clarification, guess wrong,
# or otherwise show it's missing turn_1's context? That's your evidence threads are isolated.

In [45]:
question_isolated = "Pode dar um exemplo mais específico disso?"
thread_3 = "isolated test"
turn_3_isolated = agent_v3.invoke(
    {"messages": [HumanMessage(content=question_isolated)]},
    config={"configurable": {"thread_id": thread_3}},
)

In [46]:
[i.type for i in turn_3_isolated["messages"]]

['human', 'ai', 'tool', 'ai']

In [47]:
turn_3_isolated["messages"][0].content
turn_3_isolated["messages"][1].tool_calls


'Pode dar um exemplo mais específico disso?'

[{'name': 'get_context',
  'args': {'query': 'exemplo específico'},
  'id': 'call_I05oNUgJY8qZsmVqloY3K1Ij',
  'type': 'tool_call'}]

## 7. Structured output

Add a `response_format` Pydantic model so the agent's final answer comes back as a validated object at `result["structured_response"]`, instead of just an `AIMessage.content` string you have to trust is shaped the way you expect.

**`Demo_C1`'s `Joke(BaseModel)` + `JsonOutputParser(pydantic_object=Joke)` example is only half your reference here.** The `BaseModel` field-definition mechanics still apply — but per `CLAUDE.md`, `create_agent(..., response_format=YourModel)` gives you native structured output directly; you don't chain a separate `JsonOutputParser` the way `Demo_C1` does. That parser pattern was for plain chains, not `create_agent`'s compiled graph.

The shape you're building toward:
1. Define a Pydantic `BaseModel` for what the final answer should look like.
2. Build `agent_v4` — same `model`/`tools`/`checkpointer`/`system_prompt` as `agent_v3`, plus `response_format=YourModel`.
3. Invoke it and check `result["structured_response"]`.

Questions worth thinking through before you define the schema:
- What does the caller of this agent actually need out of a "final answer" — just the answer text? Given the traceability discussion from stage 5, is there a field worth adding for *which tool* answered, or what the answer is grounded in?
- Any field worth constraining rather than leaving as a free `str` — e.g. a confidence/certainty indicator, or a boolean for "this question wasn't about the book at all"?
- `result` still has a `"messages"` key too. Once `response_format` is set, look at the message list's length/shape compared to `agent_v3`'s — does producing the structured object cost an extra model call, or does it come from the same final `AIMessage`?

Test standalone: invoke `agent_v4` with a book question, then check `type(result["structured_response"])` is your model class (not a `dict`) and that its fields actually hold sensible values — same "don't just trust it ran" discipline as every earlier stage.

In [48]:
# TODO: define a Pydantic BaseModel for the structured final answer
# HINT: see Demo_C1's Joke(BaseModel) for the field-definition mechanics (from pydantic import BaseModel, Field)
# - decide the fields (just `answer: str`? or more, per the questions above)


class MaquiModel(BaseModel):
    tool_used: Literal["get_context", "get_answer"] = Field(
    description="""O nome exato da tool que foi chamada MAIS RECENTEMENTE nesta conversa — não infira pelo conteúdo da resposta. Esta conversa pode conter múltiplas chamadas de tool de turnos anteriores; ignore-as e copie o nome literal apenas da ÚLTIMA chamada de tool (o bloco 'tool_calls: [{"name": "....", ...}]' mais próximo, imediatamente antes da resposta final atual)."""
)
    is_related_book: str = Field(
        description="a pergunta do user tem relacao como livro ?", examples=["sim", "não"]
    )
    quoted_excerpts: str = Field(
    description="""Se a tool usada foi get_answer, a tool message desta conversa contém um bloco JSON neste formato: {"question": "...", "context": "...", "answer": "..."} — localize esse bloco e copie um trecho literal do valor da chave "context". Se a tool usada foi get_context, a tool message já é o próprio texto do trecho (sem JSON) — copie diretamente dele."""
)

    question: str = Field(description="user question about the book")
    answer: str = Field(description="answer to resolve the question")


In [49]:
# TODO: build agent_v4 — same model/tools/checkpointer/system_prompt as agent_v3, plus response_format=YourModel

# TODO: build agent_v3 — same model/tools/system_prompt as agent_v2, plus checkpointer=InMemorySaver()
# HINT: reuse agent_v2's system_prompt string rather than retyping it

agent_v4 = create_agent(
    model=model,
    tools=[get_context, get_answer],
    checkpointer=InMemorySaver(),
    response_format=MaquiModel,
    system_prompt="""You are a research agent for the book "O Príncipe" by Nicolau Maquiavel.

- If the question asks what the book says about something (a fact, a definition, a passage) — use get_context to return excerpts directly from the text.
- If the question asks for analysis, interpretation, or opinion about something in the book (why, what does it mean, what's the significance) — use get_answer, which synthesizes an interpretive answer grounded in the book's content.
- get_context only retrieves, so it's cheaper; get_answer also generates an analysis, so it costs more — prefer get_context when a plain excerpt would fully answer the question.
- If the question is unrelated to this book, answer from your own knowledge without using either tool.
""",
)

In [50]:
# TODO: invoke agent_v4 with a book question (config={"configurable": {"thread_id": ...}}) and inspect:
# - result.keys() — does "structured_response" show up alongside "messages"?
# - type(result["structured_response"]) — is it your BaseModel class?
# - result["messages"] — compare length/shape to agent_v3's runs for the same kind of question

result = agent_v4.invoke(
    {"messages": [HumanMessage(content=question_1)]},
    config={"configurable": {"thread_id": thread_1}},
)

In [51]:
type(result)
result.keys()

dict

dict_keys(['messages', 'structured_response'])

In [52]:
type(result["messages"])
len(result["messages"])
[i.type for i in result["messages"]]

list

4

['human', 'ai', 'tool', 'ai']

In [53]:
result["messages"][0].content
result["messages"][1].tool_calls
result["messages"][2].content[:100]
result["messages"][3].content[:100]

"Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, incluindo a morte de Ramiro d'Orco?"

[{'name': 'get_answer',
  'args': {'query': "Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, incluindo a morte de Ramiro d'Orco?"},
  'id': 'call_nZ9VLmAiXcywHdj9RDkdh5G6',
  'type': 'tool_call'}]

'{"question": "Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da'

'{"question":"Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da '

In [54]:
# result["structured_response"].model_fields
type(result["structured_response"])
# result["structured_response"]

__main__.MaquiModel

In [55]:
result["structured_response"].tool_used
result["structured_response"].is_related_book
result["structured_response"].quoted_excerpts
result["structured_response"].question
result["structured_response"].answer[:500]

'get_answer'

'yes'

''

"Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, incluindo a morte de Ramiro d'Orco?"

'A análise das ações de César Bórgia, como descrito por Maquiavel, revela a complexidade do pensamento político maquiavélico, especialmente em relação à virtù e à fortuna. A questão central que emerge da análise é a forma como Maquiavel critica e, ao mesmo tempo, elogia a crueldade como ferramenta política nas mãos de um príncipe.\n\n**Elogio à Crueldade Estratégica:**\n\nMaquiavel elogia as ações cruéis de César Bórgia, frequentemente subordinando o moralismo a uma lógica pragmática de eficácia polí'

## Open item — `quoted_excerpts` reliability (picking this back up later)

`MaquiModel` works end to end: `tool_used` (now a `Literal`), `is_related_book`, `question`, and `answer` all come back populated and correctly typed. `quoted_excerpts` is the one field that isn't reliable yet — parking the diagnosis here so this doesn't need to be re-derived from scratch next time.

**What's actually going on — two separate problems, stacked:**

1. **Missing data (fixed).** `get_answer` originally returned only `response["answer"]`, dropping the `context` key `compose_chain` computes internally. Whenever the agent used `get_answer`, there was *nothing* for `quoted_excerpts` to draw from — a guaranteed, 100%-reproducible empty field, not a flaky one. Fixed by changing `get_answer` to return the full `response` dict (`question`, `context`, `answer`), so the excerpts now actually exist somewhere in the tool's `ToolMessage`.

2. **Unreliable extraction (still open).** Even with the data present, the model doesn't consistently pull a real quote out of it. Iterated the field's `description` several times — pointing at the literal JSON key name (`context`), then at the literal shape (`{"question": ..., "context": ..., "answer": ...}`), then branching for the `get_context` case (plain text, no JSON at all). Each version "worked" at least once and failed at least once (including 3 failures in a row right after the most explicit rewrite) — small-sample non-determinism, not a clean pass/fail signal on any one wording.

**The sharper question, not yet tested:** whether a *non-empty* `quoted_excerpts` is even genuinely grounded. `gpt-4o-mini` likely has fragments of "O Príncipe" — a centuries-old public-domain text — memorized from pretraining. A populated field could be a real extraction from the retrieved `context`, or a plausible-sounding fabrication from the model's own memory that happens to read like a Machiavelli quote. Emptiness vs. non-emptiness may be the wrong axis to debug on entirely.

**The test to run before trusting this field, next time this gets picked up:**
```python
context_text = json.loads(result["messages"][2].content).get("context", "")
result["structured_response"].quoted_excerpts in context_text
```
Run this across several fresh-kernel, single-turn invokes (mix of `get_context`- and `get_answer`-routed questions). If it's `False` even when `quoted_excerpts` is non-empty, the field has never been reliably grounded — and that's a different, harder fix (verbatim extraction under a JSON-schema-constrained decode, alongside four other fields, isn't something a more precise description can just talk its way into) than empty-vs-not was.

**Related, smaller open item:** `tool_used`'s description was rewritten to say "the most recent tool call" specifically to handle multi-turn threads where earlier turns may have called a different tool — but that scenario (same `thread_id`, two turns routing to two different tools) hasn't actually been run yet to confirm the fix works.
